# PyDI Data Integration Workflow: Products

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with product datasets to showcase the data integration pipeline from information extraction over schema and entity matching to data fusion.

The four product datasets are derived from the **WDC Products benchmark** (Web Data Commons): the same hardware products re-exported under four different attribute-naming conventions. Per-dataset provenance (including the WDC origin) is attached to each DataFrame's `attrs["provenance"]` in Part 1.

## Table of Contents

* [Part 1: Manual Schema Matching](#Part-1:-Manual-Schema-Matching)
    * [Step 1: Schema Matching](#Step-1:-Schema-Matching)
    * [Step 2: Schema Matching Evaluation](#Step-2:-Schema-Matching-Evaluation)
    * [Step 3: Data profiling (again)](#Step-3:-Data-profiling-(again))
* [Part 2: Entity Matching](#Part-2:-Entity-Matching)
    * [Step 1: Blocking](#Step-1:-Blocking)
    * [Step 2: Evaluate Blocking Against Ground Truth](#Step-2:-Evaluate-Blocking-Against-Ground-Truth)
    * [Step 3: Entity Matching with Comparators](#Step-3:-Entity-Matching-with-Comparators)
    * [Step 4: Rule Based Matcher](#Step-4:-Rule-Based-Matcher)
    * [Step 5: Evaluate Matching Against Ground Truth](#Step-5:-Evaluate-Matching-Against-Ground-Truth)
    * [Step 6: 1:1 Refinement (Greedy vs Maximum Bipartite Matching)](#Step-6:-1:1-Refinement-(Greedy-vs-Maximum-Bipartite-Matching))
    * [Step 7: ML-Based Matcher](#Step-7:-ML-Based-Matcher)
* [Part 3: Data Fusion](#Part-3:-Data-Fusion)
    * [Step 1: Define Fusion Strategy](#Step-1:-Define-Fusion-Strategy)
    * [Step 2: Run Fusion](#Step-2:-Run-Fusion)
    * [Step 3: Evaluate Data Fusion](#step-3-evaluate-data-fusion)

## Part 1: Manual Schema Matching

### Step 1: Schema Matching 

In [ ]:
import time
start_time = time.time()

import ast
import json
import pandas as pd
import logging
import os
import re
from PyDI.io import load_json
from PyDI.io import load_csv
from pathlib import Path

from PyDI.schemamatching import SchemaMappingEvaluator, SchemaTranslator
from PyDI.utils import DataProfiler
from PyDI.normalization import load_normalization_spec
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching import MaximumBipartiteMatching

from PyDI.entitymatching import MLBasedMatcher, FeatureExtractor
from sklearn.ensemble import RandomForestClassifier

from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, voting, maximum, minimum
from PyDI.fusion import DataFusionEngine
from PyDI.fusion import exact_match, numeric_tolerance_match
from PyDI.fusion import DataFusionEvaluator, DataFusionStrategy, exact_match, numeric_tolerance_match, tokenized_match


from dotenv import load_dotenv
load_dotenv()


# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
SCHEMA_DIR = INPUT_DIR / "schemamatching"



products_1_source = load_json(INPUT_DIR / "data" / "dataset_1.json",
                              name="products_1")
products_2_source = load_json(INPUT_DIR / "data" / "dataset_2.json",
                              name="products_2")
products_3_source = load_json(INPUT_DIR / "data" / "dataset_3.json",
                              name="products_3")
products_4_source = load_json(INPUT_DIR / "data" / "dataset_4.json",
                              name="products_4")

source_datasets = [products_1_source, products_2_source, products_3_source, products_4_source]
for dataset in source_datasets:
    dataset.drop(columns=["cluster_id"], inplace=True, errors="ignore")
names = ["products_1", "products_2", "products_3", "products_4"]

# Load target schema
with open(SCHEMA_DIR / "target_schema.json") as f:
    target_schema = json.load(f)

spec = load_normalization_spec(SCHEMA_DIR / "target_schema.json")
target_columns = list(spec.columns.keys())

df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

SCHEMA_MATCHES = {
    "products_1": {  # snake_case catalog feed
        "id": "id",
        "manufacturer": "brand",
        "product_name": "title",
        "product_description": "description",
        "list_price": "price",
        "currency_code": "priceCurrency",
        "product_url": "url",
        "model_name": "model",
        "manufacturer_part_number": "model_number",
        "category": "product_type",
        "gpu_chipset": "chipset_name",
        "video_memory_gb": "vram_gb",
        "capacity_gb": "storage_gb",
        "sequential_read_mb_s": "read_speed_mb_s",
        "sequential_write_mb_s": "write_speed_mb_s",
        "bus_standard": "bus_type",
        "interface": "interface_type",
        "width_millimeters": "width_mm",
        "length_millimeters": "length_mm",
        "height_millimeters": "height_mm",
        "weight_grams": "weight_g",
        "connector": "storage_connection_type",
        "memory_technology": "memory_type",
        "colour": "color",
        "form_factor": "form_factor",
    },
    "products_2": {  # camelCase web API
        "id": "id",
        "brandName": "brand",
        "name": "title",
        "descriptionText": "description",
        "priceAmount": "price",
        "currency": "priceCurrency",
        "productUrl": "url",
        "modelName": "model",
        "mpn": "model_number",
        "productCategory": "product_type",
        "chipset": "chipset_name",
        "vramGb": "vram_gb",
        "capacityGb": "storage_gb",
        "readSpeedMbps": "read_speed_mb_s",
        "writeSpeedMbps": "write_speed_mb_s",
        "busType": "bus_type",
        "interfaceType": "interface_type",
        "widthMm": "width_mm",
        "depthMm": "length_mm",
        "heightMm": "height_mm",
        "weightG": "weight_g",
        "connectionType": "storage_connection_type",
        "memoryType": "memory_type",
        "color": "color",
        "formFactor": "form_factor",
    },
    "products_3": {  # PascalCase price list
        "id": "id",
        "Brand": "brand",
        "ProductTitle": "title",
        "Details": "description",
        "Price": "price",
        "Currency": "priceCurrency",
        "Link": "url",
        "Model": "model",
        "PartNo": "model_number",
        "Type": "product_type",
        "Chipset": "chipset_name",
        "MemorySizeGB": "vram_gb",
        "CapacityGB": "storage_gb",
        "ReadMBs": "read_speed_mb_s",
        "WriteMBs": "write_speed_mb_s",
        "Bus": "bus_type",
        "Interface": "interface_type",
        "WidthMM": "width_mm",
        "LengthMM": "length_mm",
        "HeightMM": "height_mm",
        "WeightG": "weight_g",
        "Connector": "storage_connection_type",
        "MemoryType": "memory_type",
        "Colour": "color",
        "FormFactor": "form_factor",
    },
    "products_4": {  # terse ERP codes
        "id": "id",
        "mfr": "brand",
        "name": "title",
        "desc": "description",
        "amt": "price",
        "cur": "priceCurrency",
        "link": "url",
        "mdl": "model",
        "pn": "model_number",
        "cat": "product_type",
        "chip": "chipset_name",
        "vram": "vram_gb",
        "cap_gb": "storage_gb",
        "rd_mbs": "read_speed_mb_s",
        "wr_mbs": "write_speed_mb_s",
        "bus": "bus_type",
        "iface": "interface_type",
        "w_mm": "width_mm",
        "l_mm": "length_mm",
        "h_mm": "height_mm",
        "wt_g": "weight_g",
        "conn": "storage_connection_type",
        "mem": "memory_type",
        "clr": "color",
        "ff": "form_factor",
    },
}


def build_mapping(source_name, target_name, column_map):
    """Build a PyDI SchemaMapping (DataFrame) from a hand-written column map.

    The returned frame has the columns SchemaTranslator expects:
    ``source_dataset``, ``source_column``, ``target_dataset``,
    ``target_column`` and ``score``.
    """
    rows = [
        {
            "source_dataset": source_name,
            "source_column": src,
            "target_dataset": target_name,
            "target_column": tgt,
            "score": 1.0,
        }
        for src, tgt in column_map.items()
    ]
    return pd.DataFrame(
        rows,
        columns=["source_dataset", "source_column", "target_dataset", "target_column", "score"],
    )


translator = SchemaTranslator()

In [ ]:
# Build the hand-written mapping for each source and translate it into the
# shared target schema. 
mapping_1 = build_mapping("products_1", "target_schema", SCHEMA_MATCHES["products_1"])
df_final_1 = translator.translate(products_1_source, mapping=mapping_1, normalize=None)

mapping_2 = build_mapping("products_2", "target_schema", SCHEMA_MATCHES["products_2"])
df_final_2 = translator.translate(products_2_source, mapping=mapping_2, normalize=None)

mapping_3 = build_mapping("products_3", "target_schema", SCHEMA_MATCHES["products_3"])
df_final_3 = translator.translate(products_3_source, mapping=mapping_3, normalize=None)

mapping_4 = build_mapping("products_4", "target_schema", SCHEMA_MATCHES["products_4"])
df_final_4 = translator.translate(products_4_source, mapping=mapping_4, normalize=None)

mappings = {
    "products_1": mapping_1,
    "products_2": mapping_2,
    "products_3": mapping_3,
    "products_4": mapping_4,
}

# Downstream entity matching and fusion expect the shared target schema, so the
# translated frames become the working datasets from here on.
products_1_cleaned = df_final_1
products_2_cleaned = df_final_2
products_3_cleaned = df_final_3
products_4_cleaned = df_final_4

datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]

# Sanity check + provenance: every source now exposes exactly the target
# columns, and each frame still carries its WDC benchmark provenance in attrs.
for df in datasets:
    name = df.attrs["dataset_name"]
    prov = df.attrs.get("provenance")
    prov0 = prov[0] if isinstance(prov, list) else (prov or {})
    missing = [c for c in target_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in target_columns]
    print(f"{name}: {len(df)} rows, {len(df.columns)} cols, "
          f"missing={missing}, extra={extra} | "
          f"source={prov0.get('source')} ({prov0.get('naming_convention')})")

In [ ]:
# --- MaDI-Bench normalization stage: unit conversions and spellings
# that the target schema declares (units in the attribute titles, the form-factor families) plus generic
# text cleaning, applied to the translated frames before blocking; unit rescales are gated on the record's
# product type. The functions below implement the rules of the benchmark's normalization sets (see their `rule` column);
# the rules that reproduce spellings the task does not
# declare are not applied: bus_qualifier_drop, color_separator, hyphen_to_space, interface_alias, paren_unwrap, title_case_from_uniform, usb_if_fold.
import html
import re
import unicodedata
from decimal import Decimal, InvalidOperation
from typing import Any

def dec(v: Any) -> Decimal | None:
    try:
        return Decimal(str(v).strip())
    except (InvalidOperation, ValueError, AttributeError):
        return None

def op_whitespace(s: str) -> str:
    s = s.replace(" ", " ").replace(" ", " ").replace(" ", " ")
    s = s.replace("﻿", "").replace("​", "")
    return re.sub(r"\s+", " ", s).strip()

def op_markup(s: str) -> str:
    return html.unescape(re.sub(r"<[^>]+>", "", s))

def op_mojibake(s: str) -> str:
    """Undo cp1252-as-utf8 mangling ('Ã ' -> 'à'), iterated for double mangling."""
    for _ in range(3):
        try:
            t = s.encode("cp1252").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if t == s:
            break
        s = t
    return s

def op_nfkc(s: str) -> str:
    return unicodedata.normalize("NFKC", s)

def op_symbols(s: str) -> str:
    return op_whitespace(re.sub("[®™©]", "", s))

def op_title_case_from_uniform(s: str) -> str:
    """Title-case a cell whose letters are ALL upper or all lower; a mixed-case
    cell is a spelling choice and is left alone."""
    letters = [c for c in s if c.isalpha()]
    if len(letters) >= 2 and (all(c.isupper() for c in letters) or all(c.islower() for c in letters)):
        return s.title()
    return s

_BRAND_SUFFIX = ("technology", "technologies", "memory", "inc", "inc.", "corp", "corp.",
                 "corporation", "ltd", "ltd.", "llc", "co", "co.", "gmbh", "electronics")
_USB_IF = re.compile(r"\bUSB(?:-C| Type-C| Type-A)?\s*3\.[12]\s*Gen\s*1\b", re.I)
_IFACE_ALIAS = [(re.compile(r"\b(?:USB[- ]?A|Type[- ]A)\b", re.I), "USB Type-A"),
                (re.compile(r"\b(?:USB[- ]?C|Type[- ]C)\b", re.I), "USB Type-C"),
                (re.compile(r"\bmicro[- ]?usb\b", re.I), "micro-usb")]

def _op_bare_inch(s: str) -> str:
    """A lone drive-size number ('3.5', '2.5') -> 'N.N-inch' (form_factor only)."""
    return f"{s.strip()}-inch" if re.fullmatch(r"[1-5](?:\.\d)?", s.strip()) else s

def _op_binary_gb(s: str) -> str:
    d = dec(s)
    if d is not None and d >= 1024 and d % 1024 == 0:
        return str(d / 1024 * 1000)
    return s

def _op_brand_suffix_strip(s: str) -> str:
    """'Kingston Technology' -> 'Kingston': one trailing marketing/legal word from a short table."""
    toks = s.strip().split()
    if len(toks) >= 2 and toks[-1].casefold() in _BRAND_SUFFIX:
        return " ".join(toks[:-1])
    return s

def _op_dim_to_mm(s: str, ctx: dict | None = None) -> str:
    """Dimensions: GPU cards below 60 (length) / 20 (height, width) are in cm (x10);
    drives and sticks below 2 are in inches (x25.4). The notebook's wider bands (<5 inch,
    5-15 cm) would mis-scale real sizes: M.2 drives are 2.3-3.5 mm thick and sticks 6-12 mm high."""
    d = dec(s); c = ctx or {}
    if d is None or d <= 0:
        return s
    t, a = c.get("product_type", ""), c.get("attribute", "")
    if t == "GPU":
        lim = 60 if a == "length_mm" else 20
        return str((d * 10).normalize()) if d < lim else s
    if t in {"HDD", "SSD", "USB_STICK"} and d < 2:
        return str((d * Decimal("25.4")).normalize())
    return s

def _op_float_noise(s: str) -> str:
    d = dec(s)
    if d is None:
        return s
    q = d.quantize(__import__("decimal").Decimal("0.000001"))
    return str(q.normalize()) if abs(q - d) < __import__("decimal").Decimal("1e-9") and q != d else s

def _op_hyphen_collapse(s: str) -> str:
    """'A-DATA' -> 'ADATA': a hyphen between letters is dropped (brand only)."""
    return re.sub(r"(?<=[A-Za-z])-(?=[A-Za-z])", "", s)

def _op_inch(s: str) -> str:
    m = re.match(r"^(\d(?:\.\d+)?)\s*(?:\"|”|''|-?\s*[Ii]nch(?:es)?|-?\s*in\b)(.*)$", s)
    return f"{m.group(1)}-inch{m.group(2)}" if m else s

def _op_kg_to_g(s: str, ctx: dict | None = None) -> str:
    """A weight below 1 is in kg: x1000 (nothing weighs under a gram; USB sticks and M.2
    drives legitimately weigh 3 to 10 g, which a <10 threshold would mis-scale)."""
    d = dec(s)
    return str((d * 1000).normalize()) if d is not None and 0 < d < 1 else s

def op_markup(s: str) -> str:
    return html.unescape(re.sub(r"<[^>]+>", "", s))

def _op_mb_to_gb(s: str, ctx: dict | None = None) -> str:
    """VRAM of 512 or more is in MB: /1024."""
    d = dec(s)
    return str((d / 1024).normalize()) if d is not None and d >= 512 and d % 1024 == 0 else s

def op_mojibake(s: str) -> str:
    """Undo cp1252-as-utf8 mangling ('Ã ' -> 'à'), iterated for double mangling."""
    for _ in range(3):
        try:
            t = s.encode("cp1252").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if t == s:
            break
        s = t
    return s

def op_nfkc(s: str) -> str:
    return unicodedata.normalize("NFKC", s)

def op_symbols(s: str) -> str:
    return op_whitespace(re.sub("[®™©]", "", s))

def _op_tb_to_gb(s: str, ctx: dict | None = None) -> str:
    """HDD/SSD capacity below 64 is in TB: x1000 (USB sticks are never rescaled)."""
    d = dec(s)
    if d is None or (ctx or {}).get("product_type") not in {"HDD", "SSD"} or not (0 < d < 64):
        return s
    return str((d * 1000).normalize())

def op_whitespace(s: str) -> str:
    s = s.replace(" ", " ").replace(" ", " ").replace(" ", " ")
    s = s.replace("﻿", "").replace("​", "")
    return re.sub(r"\s+", " ", s).strip()

_OPS = {
    "bare_inch": _op_bare_inch,
    "binary_gb": _op_binary_gb,
    "brand_suffix_strip": _op_brand_suffix_strip,
    "dim_to_mm": _op_dim_to_mm,
    "float_noise": _op_float_noise,
    "hyphen_collapse": _op_hyphen_collapse,
    "inch_notation": _op_inch,
    "kg_to_g": _op_kg_to_g,
    "markup": op_markup,
    "mb_to_gb": _op_mb_to_gb,
    "mojibake": op_mojibake,
    "nfkc": op_nfkc,
    "symbols": op_symbols,
    "tb_to_gb": _op_tb_to_gb,
    "whitespace": op_whitespace,
}
_CTX_OPS = ['dim_to_mm', 'kg_to_g', 'mb_to_gb', 'number_locale', 'tb_to_gb']
_NUMERIC = ['height_mm', 'length_mm', 'read_speed_mb_s', 'storage_gb', 'vram_gb', 'weight_g', 'width_mm', 'write_speed_mb_s']
_RULES = {
    "brand": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols",
        "hyphen_collapse",
        "brand_suffix_strip"
    ],
    "model_number": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "product_type": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "bus_type": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "interface_type": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "form_factor": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols",
        "inch_notation",
        "bare_inch"
    ],
    "color": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "storage_connection_type": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols",
        "inch_notation"
    ],
    "memory_type": [
        "markup",
        "mojibake",
        "nfkc",
        "whitespace",
        "symbols"
    ],
    "vram_gb": [
        "float_noise",
        "mb_to_gb"
    ],
    "storage_gb": [
        "float_noise",
        "binary_gb",
        "tb_to_gb"
    ],
    "read_speed_mb_s": [
        "float_noise"
    ],
    "write_speed_mb_s": [
        "float_noise"
    ],
    "width_mm": [
        "float_noise",
        "dim_to_mm"
    ],
    "length_mm": [
        "float_noise",
        "dim_to_mm"
    ],
    "height_mm": [
        "float_noise",
        "dim_to_mm"
    ],
    "weight_g": [
        "float_noise",
        "kg_to_g"
    ]
}


def _normalize_cell(value, ops, ctx):
    if value is None or (isinstance(value, float) and value != value):
        return value
    s = str(value)
    for name in ops:
        s = _OPS[name](s, ctx) if name in _CTX_OPS else _OPS[name](s)
    return s.strip()


def normalize_products_frame(df):
    """Every rule of the table above, in order, on every cell of its attribute."""
    out = df.copy()
    types = out["product_type"].map(lambda v: str(v).strip().upper() if isinstance(v, str) else "") \
        if "product_type" in out.columns else None
    for attribute, ops in _RULES.items():
        if attribute not in out.columns or not ops:
            continue
        col = out[attribute].astype(object)
        values = [
            _normalize_cell(v, ops, {"attribute": attribute, "product_type": (types.iloc[i] if types is not None else "")})
            for i, v in enumerate(col)
        ]
        if attribute in _NUMERIC:
            # numeric attributes stay numeric (the comparators and the fusion resolvers read floats;
            # the CSV export writes them as the gold does, e.g. 1000.0)
            out[attribute] = pd.to_numeric(pd.Series(values, index=out.index), errors="coerce")
        else:
            out[attribute] = values
    out.attrs = dict(df.attrs)
    return out


_before = {n: df.copy() for n, df in zip(["products_1", "products_2", "products_3", "products_4"],
                                         [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned])}
products_1_cleaned = normalize_products_frame(products_1_cleaned)
products_2_cleaned = normalize_products_frame(products_2_cleaned)
products_3_cleaned = normalize_products_frame(products_3_cleaned)
products_4_cleaned = normalize_products_frame(products_4_cleaned)
datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]
for _name, _after in zip(_before, datasets):
    _changed = sum(int((_before[_name][a].astype(str) != _after[a].astype(str)).sum())
                   for a in _RULES if a in _after.columns)
    print(f"normalized {_name}: {_changed} cells changed by the rules")


In [ ]:
# --- MaDI-Bench normalization output: the normalized source tables, one CSV per source
# (record id + target-schema columns), the input of madi_bench.evaluation.score_normalization,
# which scores them against input/normalization/<split>.csv.
import json as _json
from pathlib import Path as _Path
_norm_dir = OUTPUT_DIR / "normalization"
_norm_dir.mkdir(parents=True, exist_ok=True)
_target = list(_json.loads((_Path(NOTEBOOK_DIR) / "input" / "schemamatching" / "target_schema.json").read_text())["properties"])
for _name, _df in {"products_1": products_1_cleaned, "products_2": products_2_cleaned, "products_3": products_3_cleaned, "products_4": products_4_cleaned}.items():
    _id_col = "id" if "id" in _df.columns else next((c for c in _df.columns if c.endswith("_id")), _df.columns[0])
    _cols = [_id_col] + [c for c in _df.columns if c in _target and c != "id" and c != _id_col]
    _df[_cols].rename(columns={_id_col: "id"}).to_csv(_norm_dir / f"{_name}.csv", index=False)
    print(f"normalized table {_name}: {len(_df)} rows, {len(_cols)} columns -> {_norm_dir / (_name + '.csv')}")


### Step 2: Schema Matching Evaluation


In [ ]:
# Evaluate the hand-written product schema mappings against the curated gold mapping.
with open(SCHEMA_DIR / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = pd.concat(mappings.values(), ignore_index=True)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


### Step 2: Data profiling

In [ ]:
profiling_datasets = [
    df_final_1,
    df_final_2,
    df_final_3,
    df_final_4
]

names = ["products_1", "products_2", "products_3", "products_4"]


total_records = sum(len(df) for df in profiling_datasets)
print(f"Total records across all profiling_datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(profiling_datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=profiling_datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print(" Attribute coverage across profiling_datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

## Part 2: Entity Matching

In [ ]:
# Set up logging
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.WARNING, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

### Step 1: Blocking

In [ ]:
out_dir_blocker =OUTPUT_DIR / "Blocking" / "standard_blocker_on_product_type"
out_dir_blocker.mkdir(parents=True, exist_ok=True)


# 3. Initialize the Star Schema Blockers
standard_blocker_p1_p2 = StandardBlocker(
    products_1_cleaned, products_2_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p2_blocking",
    id_column='id'
)

standard_blocker_p1_p3 = StandardBlocker(
    products_1_cleaned, products_3_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p3_blocking",
    id_column='id'
)

standard_blocker_p1_p4 = StandardBlocker(
    products_1_cleaned, products_4_cleaned,
    on=[ 'product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p4_blocking",
    id_column='id'
)

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
# Evaluate the blocker against the held-out test pairs for each dataset pair
test_gt_p1_p2 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod2_test.csv",
    name="test_p1_p2", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p2 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p2,
    test_pairs=test_gt_p1_p2,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod2"
)

test_gt_p1_p3 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod3_test.csv",
    name="test_p1_p3", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p3 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p3,
    test_pairs=test_gt_p1_p3,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod3"
)

test_gt_p1_p4 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod4_test.csv",
    name="test_p1_p4", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p4 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p4,
    test_pairs=test_gt_p1_p4,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod4"
)

In [ ]:
# Build a summary table from the per-pair evaluation result dicts
summary = pd.DataFrame(
    [
        {"dataset_pair": "p1_p2", **results_p1_p2},
        {"dataset_pair": "p1_p3", **results_p1_p3},
        {"dataset_pair": "p1_p4", **results_p1_p4},
    ]
)[["dataset_pair", "pair_completeness", "reduction_ratio"]]

print("Summary of Blocking Results:")
display(summary)

### Step 3: Entity Matching with Comparators

In [ ]:
# Custom hardware normalization (similar to music/companies)
# This removes noise like "GB", "TB", and punctuation to focus on model codes
def normalize_hardware_text(s: str) -> str: 
    if s is None:
        return ""
    # Remove standard units and punctuation to isolate model numbers (e.g., 980, SN850)
    s = re.sub(r"(?i)\b(gb|tb|ssd|nvme|internal|hhd|sata)\b", "", s)
    return re.sub(r"[^\w\s]|_", "", s).lower().strip()


comparators = [
    # 1. Title (Tokens) - Handles word shuffling
    StringComparator(
        column='title', 
        similarity_function='sorensen_dice', #got better results than jaccard
        tokenization='word',
        preprocess=normalize_hardware_text
    ),
    
    # 2. Brand - Switched to Jaccard since they are normalized
    StringComparator(
        column='brand',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 3. Product Type - Switched to Jaccard for strict category matching
    StringComparator(
        column='product_type',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 4. Storage GB - Essential to distinguish 500GB from 1000GB variants
    NumericComparator(
        column='storage_gb',
        method='relative_difference',
        max_difference=0.1 
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

### Step 4: Rule Based Matcher

In [ ]:
matcher = RuleBasedMatcher()

# New Weights distribution (must sum to 1.0):
# [Title_sorensen_dice, Brand_Jaccard, product_Type_Jaccard, Storage_Numeric]
# current_weights = [0.35, 0.30, 0.20, 0.15] #good balance, decent F1
# current_weights = [0.35, 0.30, 0.25, 0.10] #good for all, slightly better F1 than above (2nd best)
# current_weights = [0.30, 0.25, 0.30, 0.15] #slightly worse
current_weights = [0.30, 0.30, 0.30, 0.10] #best so far
# current_weights = [0.45, 0.15, 0.10, 0.30] #perfect precision but sinks in recall and f1
# current_weights = [0.35, 0.30, 0.30, 0.05] #worse than above
# current_weights = [0.50, 0.15, 0.30, 0.05] #massive drop in recall and F1, even if precision is perfect, so not good
# current_weights = [0.40, 0.20, 0.30, 0.10] #2nd best

current_threshold = 0.70 

# Evaluation for P1 to P2
correspondences_p1_p2 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_2_cleaned,
    candidates=standard_blocker_p1_p2, 
    comparators=comparators, 
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P3
correspondences_p1_p3 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_3_cleaned,
    candidates=standard_blocker_p1_p3,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P4
correspondences_p1_p4 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_4_cleaned,
    candidates=standard_blocker_p1_p4,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

### Step 5: Evaluate Matching Against Ground Truth

In [ ]:
# Evaluate matching results against the test ground truth for each pair
results_p1_p2 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p2,
    test_pairs=test_gt_p1_p2,
)

results_p1_p3 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p3,
    test_pairs=test_gt_p1_p3,
)

results_p1_p4 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p4,
    test_pairs=test_gt_p1_p4,
)

matching_summary = pd.DataFrame(
    [
        {"dataset_pair": "p1_p2", **results_p1_p2},
        {"dataset_pair": "p1_p3", **results_p1_p3},
        {"dataset_pair": "p1_p4", **results_p1_p4},
    ]
)[["dataset_pair", "precision", "recall", "f1", "accuracy"]]

print("Summary of Matching Results:")
display(matching_summary)

In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p2,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p2_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p2:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p2_cluster_distribution" / "p1_p2_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p2,
    out_path=cluster_details_path
)


In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p3,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p3_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p3:")
display(cluster_distribution)
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p3_cluster_distribution" / "p1_p3_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p3,
    out_path=cluster_details_path
)


In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p4,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p4_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p4:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p4_cluster_distribution" / "p1_p4_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p4,
    out_path=cluster_details_path
)

Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

### Step 6: 1:1 Refinement (Greedy vs Maximum Bipartite Matching)

In [ ]:
# Compare 1:1 refinement strategies on each candidate pool and pick the best F1 per dataset
refiners = {
    "greedy": GreedyOneToOneMatchingAlgorithm(),
    "mbm": MaximumBipartiteMatching(),
}

pairs = {
    "p1_p2": (correspondences_p1_p2, test_gt_p1_p2),
    "p1_p3": (correspondences_p1_p3, test_gt_p1_p3),
    "p1_p4": (correspondences_p1_p4, test_gt_p1_p4),
}

candidate_correspondences = {}
rows = []
for pair_name, (corr, gt) in pairs.items():
    candidate_correspondences[pair_name] = {"baseline": corr}
    rows.append({"dataset_pair": pair_name, "method": "baseline", **EntityMatchingEvaluator.evaluate_matching(correspondences=corr, test_pairs=gt)})
    for method_name, clusterer in refiners.items():
        refined = clusterer.cluster(corr)
        candidate_correspondences[pair_name][method_name] = refined
        result = EntityMatchingEvaluator.evaluate_matching(
            correspondences=refined,
            test_pairs=gt,
            out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_{method_name}",
        )
        rows.append({"dataset_pair": pair_name, "method": method_name, **result})

refinement_summary = pd.DataFrame(rows)[["dataset_pair", "method", "precision", "recall", "f1", "accuracy"]]
print("Refinement comparison (baseline vs Greedy vs Maximum Bipartite Matching):")
display(refinement_summary)

# Pick the correspondences with the best F1 per dataset pair (baseline is a valid winner)
best_per_pair = (
    refinement_summary.sort_values("f1", ascending=False)
    .drop_duplicates("dataset_pair")
    .set_index("dataset_pair")["method"]
    .to_dict()
)
print("Best method per dataset pair (by F1):", best_per_pair)

correspondences_p1_p2_refined = candidate_correspondences["p1_p2"][best_per_pair["p1_p2"]]
correspondences_p1_p3_refined = candidate_correspondences["p1_p3"][best_per_pair["p1_p3"]]
correspondences_p1_p4_refined = candidate_correspondences["p1_p4"][best_per_pair["p1_p4"]]

### Step 7: ML-Based Matcher

In [ ]:
# Train and evaluate an ML matcher on each candidate pool, then 1:1-refine the result
datasets = {
    "p1_p2": (products_1_cleaned, products_2_cleaned, standard_blocker_p1_p2,
              "prod1_to_prod2_train.csv", test_gt_p1_p2),
    "p1_p3": (products_1_cleaned, products_3_cleaned, standard_blocker_p1_p3,
              "prod1_to_prod3_train.csv", test_gt_p1_p3),
    "p1_p4": (products_1_cleaned, products_4_cleaned, standard_blocker_p1_p4,
              "prod1_to_prod4_train.csv", test_gt_p1_p4),
}

extractor = FeatureExtractor(comparators)
ml_refiner = GreedyOneToOneMatchingAlgorithm()  # not MaximumBipartiteMatching, which ignores the scores
ml_results = {}
ml_correspondences = {}
for pair_name, (df_left, df_right, blocker, train_file, gt) in datasets.items():
    train_pairs = load_csv(
        OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / train_file,
        add_index=False,
    )
    train_features = extractor.create_features(
        df_left=df_left, df_right=df_right, pairs=train_pairs,
        id_column='id', labels=train_pairs['label'],
    )
    X_train = train_features.drop(['label', 'id1', 'id2'], axis=1)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, train_features['label'])

    matcher = MLBasedMatcher(extractor)
    raw_ml = matcher.match(
        df_left=df_left, df_right=df_right, candidates=blocker,
        id_column='id', trained_classifier=clf,
        threshold=0.45, use_probabilities=True,
    )
    # Refine ML output to 1:1 so it can be fused without producing mega-clusters
    ml_correspondences[pair_name] = ml_refiner.cluster(raw_ml)
    ml_results[pair_name] = EntityMatchingEvaluator.evaluate_matching(
        correspondences=ml_correspondences[pair_name], test_pairs=gt,
    )

# Compare ML matcher against the best rule-based result per dataset (from refinement_summary)
best_rule_based = (
    refinement_summary.sort_values("f1", ascending=False)
    .drop_duplicates("dataset_pair")
    .set_index("dataset_pair")
)

comparison_rows = []
for pair_name, ml_res in ml_results.items():
    rule_row = best_rule_based.loc[pair_name]
    comparison_rows.append({
        "dataset_pair": pair_name,
        "rule_method": rule_row["method"],
        "rule_f1": rule_row["f1"],
        "ml_f1": ml_res["f1"],
        "f1_delta": ml_res["f1"] - rule_row["f1"],
        "rule_precision": rule_row["precision"],
        "ml_precision": ml_res["precision"],
        "rule_recall": rule_row["recall"],
        "ml_recall": ml_res["recall"],
    })

ml_vs_rule = pd.DataFrame(comparison_rows)
ml_vs_rule["winner"] = ml_vs_rule.apply(
    lambda row: "ml" if row["ml_f1"] > row["rule_f1"] else row["rule_method"], axis=1,
)
print("ML matcher (1:1 refined) vs best rule-based method per dataset:")
display(ml_vs_rule)

winners = ml_vs_rule.set_index("dataset_pair")["winner"].to_dict()
print("Final candidate pool per dataset pair:", winners)

fusion_safe_refiner = GreedyOneToOneMatchingAlgorithm()  # not MaximumBipartiteMatching, which ignores the scores
def pick(pair_name):
    method = winners[pair_name]
    chosen = ml_correspondences[pair_name] if method == "ml" else candidate_correspondences[pair_name][method]
    # baseline is raw N:M and would collapse fusion into mega-clusters; force 1:1 (no-op for already-refined pools)
    return fusion_safe_refiner.cluster(chosen) if method == "baseline" else chosen

correspondences_p1_p2_refined = pick("p1_p2")
correspondences_p1_p3_refined = pick("p1_p3")
correspondences_p1_p4_refined = pick("p1_p4")

print(
    "Refined link counts:",
    {
        "p1_p2": len(correspondences_p1_p2_refined),
        "p1_p3": len(correspondences_p1_p3_refined),
        "p1_p4": len(correspondences_p1_p4_refined),
    },
)

## Part 3: Data Fusion

### we fist split our data into test and validation 

In [ ]:
# Set the base directory to where the notebook is
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "input" / "fusion"

# Load using the short, relative path
val_set = pd.read_csv(DATA_DIR / 'fusion_validation_set.csv')
test_set = pd.read_csv(DATA_DIR / 'fusion_test_set.csv')

def normalize_source_ids(value):
    """Return a clean list of source ids from CSV list/string formats."""
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith('['):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
        except json.JSONDecodeError:
            pass
    return [part.strip() for part in text.split(',') if part.strip()]

def source_ids_to_csv(value):
    return ','.join(normalize_source_ids(value))

def format_product_source_id(dataset_name, record_id):
    dataset_name = str(dataset_name).strip()
    record_text = str(record_id).strip()
    if record_text.endswith('.0'):
        record_text = record_text[:-2]
    return f"{dataset_name}_{record_text}"

for df in [val_set, test_set]:
    df['source_ids'] = df['source_ids'].map(source_ids_to_csv)

print(f"Loaded Validation: {len(val_set)} rows")
print(f"Loaded Test: {len(test_set)} rows")


#### we need to see which file is more trustworthy so we use or fusion validation set to check

In [ ]:
def hardware_strict_spec_match(fused_value, expected_value):
    """Compare numeric tokens strictly; prevents 'PCIE x8' matching 'PCIE x16'."""
    if pd.isna(fused_value) or pd.isna(expected_value):
        return False
    f = str(fused_value).lower()
    e = str(expected_value).lower()
    if re.findall(r'\d+', f) != re.findall(r'\d+', e):
        return False
    f_clean = re.sub(r'[^a-z0-9]', '', f)
    e_clean = re.sub(r'[^a-z0-9]', '', e)
    return e_clean in f_clean or f_clean in e_clean

audit_strategy = DataFusionStrategy('validation_audit')

for attr in ['brand', 'product_type', 'model_number']:
    audit_strategy.add_evaluation_function(attr, exact_match)

num_attrs = ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s',
             'width_mm', 'length_mm', 'height_mm', 'weight_g']
for attr in num_attrs:
    audit_strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

tech_attrs = ['bus_type', 'interface_type', 'chipset_name',
              'storage_connection_type', 'memory_type', 'form_factor']
for attr in tech_attrs:
    audit_strategy.add_evaluation_function(attr, hardware_strict_spec_match)

evaluator = DataFusionEvaluator(audit_strategy)

sources = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]
source_names = ["products_1", "products_2", "products_3", "products_4"]

audit_summary = []
for df_source, source_name in zip(sources, source_names):
    source_eval = df_source.copy()
    source_eval['source_id'] = source_eval['id'].map(lambda record_id: format_product_source_id(source_name, record_id))
    relevant_gt = val_set[
        val_set['source_ids'].map(
            lambda ids: any(source_id.startswith(f"{source_name}_") for source_id in normalize_source_ids(ids))
        )
    ]
    results = evaluator.evaluate(
        fused_df=source_eval,
        fused_id_column='source_id',
        gold_df=relevant_gt,
        gold_id_column='source_ids',
    )
    audit_summary.append({
        'Source': source_name,
        'Overall_Accuracy': results.get('overall_accuracy'),
        'Details': results.get('per_attribute_scores'),
    })

report = pd.DataFrame(audit_summary).set_index('Source')
print("Validation audit results:")
display(report[['Overall_Accuracy']])


#### based on trust scores. we use this order of trust

In [ ]:
# Set P1 ID as the master ID for the fused records
products_1_cleaned["p1_id"] = products_1_cleaned["id"]

# Trust scores (1 = lowest, 3 = highest)
products_1_cleaned.attrs["trust_score"] = 3
products_2_cleaned.attrs["trust_score"] = 2
products_3_cleaned.attrs["trust_score"] = 1
products_4_cleaned.attrs["trust_score"] = 2

all_correspondences = pd.concat([
    correspondences_p1_p2_refined,
    correspondences_p1_p3_refined,
    correspondences_p1_p4_refined,
], ignore_index=True)

# concat can upcast int id columns to float when frames differ in dtype; cast back
all_correspondences['id1'] = all_correspondences['id1'].astype('int64')
all_correspondences['id2'] = all_correspondences['id2'].astype('int64')

print(f"Total refined links for fusion: {len(all_correspondences):,}")

### Step 1: Define Fusion Strategy

In [ ]:
strategy = DataFusionStrategy('hardware_fusion_strategy')

# 1. Identity
for attr in ['brand', 'product_type', 'model_number']:
    strategy.add_attribute_fuser(attr, voting)

# 2. Performance Specs (Numbers)
for attr in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    strategy.add_attribute_fuser(attr, minimum) #i think min could be better. lets see

# 3. Technical & Dimensions (Using Trust)
tech_and_dim_attrs = [
    'chipset_name', 'bus_type', 'interface_type', 'storage_connection_type', 
    'memory_type', 'form_factor', 'width_mm', 'length_mm', 'height_mm', 'weight_g'
]
for attr in tech_and_dim_attrs:
    strategy.add_attribute_fuser(attr, prefer_higher_trust, trust_key="trust_score")



### Step 2: Run Fusion

In [ ]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json', 
                          debug_file=OUTPUT_DIR / "data_fusion" / "hardware_fusion_debug.jsonl")

fused = engine.run(
    datasets=[products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False, # Set to True if you want to keep products that didn't have matches
)   

print(f"Final Fused Rows: {len(fused):,}")
display(fused.head(1)) #maybe lower to 1 because the ouput is huge

### Step 3: Evaluate Data Fusion

In [ ]:
# --- MaDI-Bench fusion evaluation: the public evaluation package scores the
# fused table on the validation and the test split -- strict comparators on the v2 gold set,
# all-gold accuracy (a gold record the pipeline did not produce counts as wrong) with its coverage.
# This is the scorer the benchmark's reference numbers were produced with; the notebook's own
# comparator registrations above are not used for scoring.
import json as _json, os as _os, sys as _sys
from pathlib import Path as _Path
try:
    from madi_bench.evaluation import score_fusion as _score_fusion
except ImportError:
    _sys.path.insert(0, _os.environ.get("MADI_BENCH_ROOT") or str(_Path(NOTEBOOK_DIR).resolve().parents[2]))
    from madi_bench.evaluation import score_fusion as _score_fusion
_fused_out = fused.copy()
# the notebook's own column names for target-schema attributes (the same aliases the benchmark's
# rescoring applies, reproduction/scoring/rescore.py): scoring is by target-schema name
for _src, _dst in ():
    if _src in _fused_out.columns and _dst not in _fused_out.columns:
        _fused_out = _fused_out.rename(columns={_src: _dst})
_fusion_dir = OUTPUT_DIR / "data_fusion"
_fusion_dir.mkdir(parents=True, exist_ok=True)
_fused_out.to_csv(_fusion_dir / "fused.csv", index=False)
_scores = {}
for _split in ("validation", "test"):
    _r = _score_fusion(NOTEBOOK_DIR, _fused_out, split=_split)
    _scores[_split] = _r
    print(f"{_split}: all-gold accuracy {_r['overall_accuracy_all_gold']:.4f} "
          f"(evaluated {_r['overall_accuracy_evaluated']:.4f} on {_r['n_gold_evaluated']}/{_r['n_gold']} gold records, "
          f"attribute cell coverage {_r['attribute_cell_coverage']:.2f})")
    for _a, _v in sorted(_r["per_attribute_accuracy_all_gold"].items()):
        print(f"    {_a:28s} {_v:.4f}")
(_fusion_dir / "fusion_scores_v2.json").write_text(_json.dumps(_scores, indent=1, default=str))


In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")